# 🏠 Airbnb Market Analysis in European Cities

This project aims to analyze Airbnb listing data across multiple European cities to understand the factors influencing pricing, customer satisfaction, and listing characteristics.

The dataset includes listings from different cities, separated into weekdays and weekends, allowing for comparative analysis of pricing behavior across time and location.

## 🎯 Objective

The main objective of this project is to identify and understand the most important drivers of Airbnb prices across European cities.

Through exploratory data analysis and business-focused insights, this project seeks to:

- Identify the main factors influencing listing prices
- Understand how location affects pricing
- Compare pricing behavior across cities and day types
- Analyze the relationship between price and guest satisfaction
- Evaluate the impact of property characteristics on price
- Provide practical recommendations for Airbnb hosts and travelers

## ❓ Key Questions

### 💰 Pricing Drivers
1. What are the strongest factors influencing Airbnb prices across European cities?
2. How do location, room type, and property size affect listing prices?

### 📍 Location & Accessibility
3. How does distance from the city center impact Airbnb pricing?
4. Does proximity to metro stations influence listing prices?

### 🏙️ Cross-City Market Intelligence
5. Which cities are the most expensive, and what factors explain these differences?
6. Which cities offer the best value for money based on price and guest satisfaction?

### 🏠 Property Segmentation
7. How do room types (entire home, private room, shared room) affect pricing?

8. Are larger properties (more bedrooms / higher capacity) always priced higher?

### ⭐ Customer Satisfaction
9. Are higher-priced listings associated with better guest satisfaction?
10. Which factors contribute most to higher guest satisfaction?

### 📆 Time-Based Pricing Strategy
11. How much of a weekend price premium exists across different cities?
12. Which cities show the strongest weekend price increase?

### 💡 Business Recommendations
13. What pricing strategies could Airbnb hosts adopt based on city, location, and property type?
14. How can hosts improve competitiveness while maintaining strong guest satisfaction?



## 1. Load Cleaned Dataset

We start by loading the cleaned dataset generated during the data wrangling phase.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go


df = pd.read_csv(r"..\Data\processed\airbnb_cleaned.csv")

df.head()

,price,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,rating,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,194.033698,Private room,False,True,2,False,1,0,10.0,93.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,amsterdam,weekday
1,344.245776,Private room,False,True,4,False,0,0,8.0,85.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,amsterdam,weekday
2,264.101422,Private room,False,True,2,False,0,1,9.0,87.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,amsterdam,weekday
3,433.529398,Private room,False,True,4,False,0,1,9.0,90.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,amsterdam,weekday
4,485.552926,Private room,False,True,2,True,0,0,10.0,98.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,amsterdam,weekday


In [3]:
## For exporting images for presentaion 
!pip install kaleido

## 2. Dataset Overview

We inspect the structure and general properties of the dataset.

In [4]:
df.shape
df.info()
df.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   price               51707 non-null  float64
 1   room_type           51707 non-null  str    
 2   room_shared         51707 non-null  bool   
 3   room_private        51707 non-null  bool   
 4   person_capacity     51707 non-null  int64  
 5   host_is_superhost   51707 non-null  bool   
 6   multi               51707 non-null  int64  
 7   biz                 51707 non-null  int64  
 8   cleanliness_rating  51707 non-null  float64
 9   rating              51707 non-null  float64
 10  bedrooms            51707 non-null  int64  
 11  dist                51707 non-null  float64
 12  metro_dist          51707 non-null  float64
 13  attr_index          51707 non-null  float64
 14  attr_index_norm     51707 non-null  float64
 15  rest_index          51707 non-null  float64
 16  rest_index_norm

,count,mean,std,min,25%,50%,75%,max
price,51707.0,279.879591,327.948386,34.779339,148.752174,211.343089,319.694287,18545.450285
person_capacity,51707.0,3.161661,1.298545,2.000000,2.000000,3.000000,4.000000,6.000000
multi,51707.0,0.291353,0.454390,0.000000,0.000000,0.000000,1.000000,1.000000
biz,51707.0,0.350204,0.477038,0.000000,0.000000,0.000000,1.000000,1.000000
cleanliness_rating,51707.0,9.390624,0.954868,2.000000,9.000000,10.000000,10.000000,10.000000
rating,51707.0,92.628232,8.945531,20.000000,90.000000,95.000000,99.000000,100.000000
bedrooms,51707.0,1.158760,0.627410,0.000000,1.000000,1.000000,1.000000,10.000000
dist,51707.0,3.191285,2.393803,0.015045,1.453142,2.613538,4.263077,25.284557
metro_dist,51707.0,0.681540,0.858023,0.002301,0.248480,0.413269,0.737840,14.273577
attr_index,51707.0,294.204105,224.754123,15.152201,136.797385,234.331748,385.756381,4513.563486


The dataset contains information about Airbnb listings across multiple European cities, including pricing, location, property characteristics, and guest ratings.

Key features include:
- Price and room type
- Location indicators (distance, attraction index)
- Guest experience (ratings, cleanliness)
- Listing characteristics (capacity, bedrooms)

## 3. Outlier Detection

Extreme prices can distort analysis.  
We use the IQR method to identify and remove outliers.

In [5]:
q1 = df["price"].quantile(0.25)
q3 = df["price"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Lower:", lower_bound)
print("Upper:", upper_bound)

Lower: -107.660995498034
Upper: 576.1074557199221


## 5. Remove Outliers

We create a filtered dataset without extreme price values to ensure more reliable insights.

In [6]:
df_filtered = df[
    (df["price"] >= lower_bound) &
    (df["price"] <= upper_bound)
].copy()

df_filtered.shape

(48045, 21)

## 6. Clean Price Distribution

After removing outliers, the price distribution becomes clearer and more interpretable.

For many of the final presentation insights, **median** was used instead of **average**.

This is because Airbnb price data is skewed and can include extreme values. 
 
Median provides a more robust measure of the typical listing price, while average can be more affected by unusually expensive listings.

Most listings are concentrated roughly between €100 and €250 per night, with a right-skewed distribution driven by fewer expensive listings.

In [7]:
mean_price = df_filtered["price"].mean()
median_price = df_filtered["price"].median()

fig = px.histogram(
    df_filtered,
    x="price",
    nbins=40,
    marginal="box",
    opacity=0.85,
    color_discrete_sequence=["#4C78A8"],
    title="Airbnb Price Distribution per Night (€)"
)

fig.add_vline(
    x=mean_price,
    line_dash="dash",
    line_color="#E45756",
    annotation_text=f"Mean = {mean_price:.1f}€",
    annotation_position="top right"
)

fig.add_vline(
    x=median_price,
    line_dash="dash",
    line_color="#2E8B57",
    annotation_text=f"Median = {median_price:.1f}€",
    annotation_position="top left"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=1100,
    height=550,
    bargap=0.03,
    xaxis_title="Price (€ per night)",
    yaxis_title="Number of Listings",
    font=dict(size=13),
    showlegend=False
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor="rgba(0,0,0,0.08)")

fig.show()

## 7. Median Price by City

We use median instead of mean to avoid distortion from extreme values.

In [8]:
city_price = (
    df_filtered
    .groupby("city")["price"]
    .median()
    .reset_index()
    .sort_values("price", ascending=True)
)

In [9]:
fig = px.bar(
    city_price,
    x="price",
    y="city",
    orientation="h",
    title="Which Cities Are the Most Expensive?",
    text=city_price["price"].round(0)
)

fig.update_traces(
    marker_color="#4C78A8"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Median Price (€ per night)",
    yaxis_title=""
)

fig.show()

In [10]:
fig = px.bar(
    city_price,
    x="price",
    y="city",
    orientation="h",
    color="price",
    color_continuous_scale="Blues",
    title="Median Airbnb Price by City (€ per Night)",
    text=city_price["price"].round(1)
)

fig.update_traces(
    textposition="outside",
    marker_line_width=0
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=1100,
    height=550,
    xaxis_title="Median Price (€ per night)",
    yaxis_title="City",
    font=dict(size=13),
    coloraxis_showscale=False,
    margin=dict(l=80, r=40, t=70, b=50)
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)"
)

fig.update_yaxes(
    categoryorder="total ascending"
)

fig.show()



## 8. Price Distribution by City

Box plots reveal:
- spread of prices
- median
- presence of remaining outliers

In [11]:
fig = px.box(
    df_filtered,
    x="city",
    y="price",
    color="city",
    title="Price Distribution by City"
)
fig.show()



Cities like Amsterdam, Paris, and London show a wider spread in prices, indicating a mix of budget and luxury listings.

In contrast, cities like Athens and Budapest have more concentrated price ranges, suggesting a more uniform market.

## 9. Price by Room Type

We analyze how different room types affect pricing.

In [12]:
room_price = (
    df_filtered.groupby("room_type")["price"]
    .median()
    .reset_index()
)

room_price["label"] = room_price["price"].round(0).astype(int).astype(str) + " €"

In [13]:
fig = px.bar(
    room_price,
    x="room_type",
    y="price",
    text="label",
    color="room_type",
    color_discrete_map={
        "Entire home/apt": "#1f77b4",
        "Private room": "#ff7f0e",
        "Shared room": "#2ca02c"
    },
    title="💰 Price Difference by Room Type"
)

# 👇 Bar Width
fig.update_traces(
    textposition="outside",
    textfont=dict(size=14),
    width=0.5  #  Bar Width thinning
)

fig.update_layout(
    template="plotly_white",
    title=dict(x=0.5, font=dict(size=20)),
    xaxis_title="",
    yaxis_title="Median Price (€ per night)",
    font=dict(size=14),
    showlegend=False,
    width=750,     
    height=450,  
    margin=dict(t=70, b=40)
)


fig.update_yaxes(gridcolor="rgba(0,0,0,0.1)")

fig.show()


### Insight

Entire homes are significantly more expensive than other room types, reflecting the premium placed on privacy and full property access.

On average:
- Entire homes cost ~40% more than private rooms  
- Entire homes cost ~85% more than shared rooms  
- Private rooms cost ~30% more than shared rooms  

This highlights how space and exclusivity are major drivers of Airbnb pricing.

## 10. Capacity & Bedrooms

This section examines how listing size is associated with Airbnb price.

The analysis focuses on:
- guest capacity
- number of bedrooms

Median price is used to reduce the effect of extreme values and to make the comparison more robust.

In [14]:
cap_summary = (
    df_filtered.groupby("person_capacity")["price"]
    .median()
    .reset_index()
)

bed_summary = (
    df_filtered.groupby("bedrooms")["price"]
    .median()
    .reset_index()
)

cap_summary
bed_summary

,bedrooms,price
0,0,220.432466
1,1,187.735254
2,2,252.878642
3,3,307.692308
4,4,335.967450
5,5,233.307463
6,8,245.184506
7,9,130.863039
8,10,77.743902


Here, guest capacity does not represent the actual number of booked guests.
It represents the maximum number of people the listing can accommodate.
So this chart shows that larger-capacity listings tend to have higher total prices.

In [15]:
df_filtered["price_per_guest"] = df_filtered["price"] / df_filtered["person_capacity"]

ppg_summary = (
    df_filtered.groupby("person_capacity")["price_per_guest"]
    .median()
    .reset_index()
)

In [16]:
ppg_summary = ppg_summary.sort_values("person_capacity").copy()
ppg_summary["label"] = ppg_summary["price_per_guest"].round(1).astype(str) + "€"

fig = px.bar(
    ppg_summary,
    x="person_capacity",
    y="price_per_guest",
    text="label",
    title="Value per Guest by Capacity"
)

fig.update_traces(
    textposition="outside",
    marker_color=["#1D4ED8", "#3B82F6", "#60A5FA", "#93C5FD", "#BFDBFE"],
    marker_line_width=0,
    width=0.58
)

fig.update_layout(
    template="simple_white",
    title_x=0.5,
    width=860,
    height=540,
    xaxis_title="Guest Capacity",
    yaxis_title="Median Price per Guest (€)",
    font=dict(size=14, color="#243B53"),
    showlegend=False,
    bargap=0.35,
    margin=dict(t=90, b=60, l=70, r=40),
    annotations=[
        dict(
            text="Median listing price divided by maximum guest capacity",
            x=0.5, y=1.06, xref="paper", yref="paper",
            showarrow=False, font=dict(size=13, color="#5B657A")
        ),
        dict(
            text="<b>Key insight:</b><br>Higher-capacity listings offer<br>lower median price per guest.",
            x=0.74, y=0.93, xref="paper", yref="paper",
            showarrow=False,
            align="left",
            bgcolor="rgba(239,244,255,0.95)",
            bordercolor="rgba(29,78,216,0.18)",
            borderwidth=1,
            borderpad=8,
            font=dict(size=12, color="#243B53")
        )
    ]
)

fig.update_yaxes(gridcolor="rgba(15,23,42,0.08)", zeroline=False)
fig.update_xaxes(showgrid=False)

fig.show()


### Insight

Although larger listings tend to have higher total prices, their median price per guest decreases as guest capacity increases.

This indicates that bigger properties may provide better value-for-capacity than smaller ones.

Median price per guest decreases as guest capacity increases.

This suggests that larger listings may offer better value relative to the number of guests they can accommodate, even if their total price is higher.

## 11. Weekend Premium by City

We calculate how much prices increase during weekends per city.

In [17]:
weekend = (
    df_filtered.groupby(["city", "day_type"])["price"]
    .median()
    .unstack()
)

weekend["premium_pct"] = (
    (weekend["weekend"] - weekend["weekday"]) / weekend["weekday"] * 100
)

weekend = weekend.reset_index().sort_values("premium_pct", ascending=False)

weekend

day_type,city,weekday,weekend,premium_pct
0,amsterdam,350.104281,387.012865,10.542169
4,budapest,147.342200,159.796964,8.452951
6,london,225.546226,238.011242,5.526590
8,rome,178.851144,184.462161,3.137255
9,vienna,203.819274,208.727766,2.408257
5,lisbon,223.030019,225.609756,1.156677
3,berlin,184.981771,187.085164,1.137081
1,athens,127.715417,127.715417,0.000000
7,paris,289.868580,289.868580,0.000000
2,barcelona,197.593502,196.895292,-0.353357


In [18]:
weekend = weekend.sort_values("premium_pct", ascending=False)

weekend["label"] = weekend["premium_pct"].round(1).astype(str) + "%"

fig = px.bar(
    weekend,
    x="city",
    y="premium_pct",
    color="premium_pct",
    color_continuous_scale="Viridis",
    title="Weekend Price Premium by City (%)",
    text="label"
)

fig.update_traces(
    textposition="outside",
    textfont=dict(color="black", size=13)
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=950,
    height=500,
    xaxis_title="City",
    yaxis_title="Weekend Price Increase (%)",
    coloraxis_showscale=False,
    font=dict(size=14)
)

fig.update_xaxes(tickfont=dict(size=14, color="black"))
fig.update_yaxes(
    tickfont=dict(size=14, color="black"),
    range=[weekend["premium_pct"].min() - 1, weekend["premium_pct"].max() + 1]
)

fig.show()


### Insight

Weekend pricing behavior varies significantly across cities.

Cities like Amsterdam and Budapest show the highest weekend price increases, indicating strong demand during weekends and a clear opportunity for hosts to charge premium rates.

On the other hand, cities such as Athens and Paris show little to no price change, suggesting more stable demand throughout the week.

Interestingly, some cities even show slight price decreases during weekends, which may indicate promotional pricing strategies or lower weekend demand.

Overall, weekend premiums highlight how demand patterns differ across markets and reflect varying levels of tourism activity.

## 12. Value for Money

We evaluate how much rating a listing offers relative to its price.

We define a value score as:

Value = Guest Satisfaction / Price

This helps identify which cities offer better experiences relative to their cost.

In [19]:
value_city = (
    df_filtered.groupby("city")
    .agg(
        price=("price", "median"),
        rating=("rating", "median")
    )
    .reset_index()
)

# The higher the value score the better 
value_city["value_score"] = value_city["rating"] / value_city["price"]

In [35]:
# Arranging the data from the higher to the lower
value_city = value_city.sort_values("value_score", ascending=False)

fig = px.bar(
    value_city,
    x="city",
    y="value_score",
    text=value_city["value_score"].round(2),
    color="value_score",
    color_continuous_scale="Viridis",
    title="Value for Money by City (Higher = Better)"
)

fig.update_traces(
    textposition="outside",
    textfont=dict(size=13, color="black")
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=950,
    height=500,
    yaxis_title="Value Score",
    xaxis_title="City",
    coloraxis_showscale=False,
    font=dict(size=14)
)

#  Makine the city names more clearer and darker
fig.update_xaxes(
    tickfont=dict(size=13, color="black")
)

fig.update_yaxes(
    tickfont=dict(size=13, color="black"),
    gridcolor="rgba(0,0,0,0.1)"
)

fig.show()

fig.write_image("Value for Money by City (Higher = Better).png", scale=3)


### Insight

Cities like Athens and Budapest offer the best value for money, combining lower prices with high guest satisfaction.

On the other hand, cities like Amsterdam and Paris have lower value scores, indicating that higher prices do not necessarily translate into better guest experiences.

This suggests that travelers can achieve better value in less expensive cities.

## 13. Distance vs Price

We examine how distance from city center or metro affects price.

In [21]:
bins = [0, 1, 2, 5, 10, max(df_filtered["dist"].max(), df_filtered["metro_dist"].max())]
labels = ["0-1 km", "1-2 km", "2-5 km", "5-10 km", "10+ km"]

df_filtered["dist_band"] = pd.cut(df_filtered["dist"], bins=bins, labels=labels)
df_filtered["metro_band"] = pd.cut(df_filtered["metro_dist"], bins=bins, labels=labels)

In [22]:

# Collecting center data prices
center_price = (
    df_filtered.groupby("dist_band")["price"]
    .median()
    .reset_index()
)
center_price.columns = ["band", "median_price"]
center_price["type"] = "City Center"

In [23]:

# Collecting metro data prices

metro_price = (
    df_filtered.groupby("metro_band")["price"]
    .median()
    .reset_index()
)
metro_price.columns = ["band", "median_price"]
metro_price["type"] = "Metro"

In [24]:
# Merging both of the collected data in one table 
distance_compare = pd.concat([center_price, metro_price], ignore_index=True)

In [25]:
distance_compare["label"] = distance_compare["median_price"].round(1)



In [26]:
fig = px.bar(
    distance_compare,
    x="band",
    y="median_price",
    color="type",
    barmode="group",
    text="label",
    color_discrete_map={"City Center": "#4C6EF5", "Metro": "#12B886"},
    title="How Median Price Changes with Distance from the Center and the Metro"
)

fig.update_traces(
    textposition="outside",
    textfont=dict(size=11, color="black"),
    width=0.36
)


fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    title_font=dict(size=18), 
    xaxis_title="Distance Band",
    yaxis_title="Median Price (€ per night)",
    width=900,
    height=500,
    font=dict(size=13),
    legend_title_text="Distance Type",
    margin=dict(t=70, b=50, l=60, r=30)
)

fig.update_xaxes(
    tickfont=dict(size=12, color="black")
)

fig.update_yaxes(
    tickfont=dict(size=12, color="black"),
    gridcolor="rgba(0,0,0,0.07)",
    range=[0, 280]  # 👈For making the axis clearer
)

fig.show()



In [27]:
df_filtered["dist_band"].value_counts()
df_filtered["metro_band"].value_counts()

metro_band
0-1 km     39573
1-2 km      5723
2-5 km      2299
5-10 km      445
10+ km         5
Name: count, dtype: int64

While prices generally decrease with distance, we observe a spike in the 10+ km metro band.
This is likely due to a smaller number of listings and the presence of premium properties.
This shows that distance is not the only factor affecting price.

## 14. What Drives Airbnb Prices?

This section explores which features have the strongest relationship with price.

Understanding these drivers helps explain why listings are priced differently across cities and properties.

In [28]:
selected_features = [
    "attr_index_norm",
    "rest_index_norm",
    "person_capacity",
    "bedrooms",
    "dist",
    "metro_dist",
    "room_private",
    "room_shared"
]

name_map = {
    "attr_index_norm": "Attraction Score",
    "rest_index_norm": "Restaurant Score",
    "person_capacity": "Capacity",
    "bedrooms": "Bedrooms",
    "dist": "Distance to Center",
    "metro_dist": "Distance to Metro",
    "room_private": "Private Room Listing",
    "room_shared": "Shared Room Listing"
}

In [29]:

corr_filtered = (
    df_filtered[selected_features + ["price"]]
    .corr(numeric_only=True)["price"]
    .drop("price")
    .reset_index()
)

corr_filtered.columns = ["feature", "correlation"]
corr_filtered["feature"] = corr_filtered["feature"].map(name_map)
corr_filtered = corr_filtered.sort_values("correlation")

In [30]:
fig = px.bar(
    corr_filtered,
    x="correlation",
    y="feature",
    orientation="h",
    color="correlation",
    color_continuous_scale="RdBu",
    text=corr_filtered["correlation"].round(2),
    title="Key Factors Associated with Price"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    marker_line_width=0
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=1050,
    height=560,
    xaxis_title="Correlation with Price",
    yaxis_title="",
    coloraxis_showscale=False,
    font=dict(size=13),
    margin=dict(l=80, r=50, t=70, b=50)
)

fig.update_xaxes(
    tickfont=dict(size=12, color="black"),
    range=[-0.33, 0.43]
)

fig.update_yaxes(
    tickfont=dict(size=12, color="black"),
    categoryorder="array",
    categoryarray=[
        "Private Room Listing",
        "Shared Room Listing",   
    ]
)

fig.add_vline(x=0, line_dash="dash", line_color="gray")

fig.show()

fig.write_image("key_factors_associated_with_price.png", scale=3)

### Insight

This chart shows the most interpretable drivers of Airbnb price. Larger properties, such as those with higher capacity and more bedrooms, tend to have higher prices, while private rooms and more distant listings tend to be cheaper.

We can see that capacity and number of bedrooms have the strongest positive impact on price.
This means larger properties tend to be more expensive.

On the other hand, private rooms and shared rooms are negatively correlated with price, meaning they are generally cheaper compared to full apartments.

Overall, price is mainly driven by property size and type, followed by location.

## 16. Superhost vs Regular Host
In this section, I compare listings hosted by Superhosts and Regular Hosts.

The goal is to examine whether the Superhost badge is associated with:
- higher prices
- better guest ratings
- stronger cleanliness scores

Here, `host_is_superhost = True` represents Superhosts, while `False` represents Regular Hosts.

In [31]:
superhost_summary = (
    df_filtered.groupby("host_is_superhost")
    .agg(
        median_price=("price", "median"),
        median_rating=("rating", "median"),
        median_cleanliness=("cleanliness_rating", "median"),
        listings=("price", "size")
    )
    .reset_index()
)

superhost_summary["host_is_superhost"] = superhost_summary["host_is_superhost"].map({
    True: "Superhost",
    False: "Regular Host"
})

superhost_summary

,host_is_superhost,median_price,median_rating,median_cleanliness,listings
0,Regular Host,205.790353,93.0,9.0,35489
1,Superhost,191.221616,97.0,10.0,12556


In [32]:
fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Host Type", "Median Price (€)", "Median Rating", "Listings"],
        fill_color="#1f3b73",
        font=dict(color="white", size=14),
        align="center"
    ),
    cells=dict(
        values=[
            superhost_summary["host_is_superhost"],
            superhost_summary["median_price"].round(2),
            superhost_summary["median_rating"].round(0),
            superhost_summary["listings"]
        ],
        fill_color="white",
        align="center",
        font=dict(size=13)
    )
)])

fig.update_layout(
    title="Superhost vs Regular Host",
    title_x=0.5,
    template="plotly_white",
    width=1000,
    height=350
)

fig.show()


### Initial Insight
Superhosts are not more expensive in median price, but they show better rating and cleanliness performance than Regular Hosts.

## 18. Business Hosts vs Individual Hosts

In this section, I compare business hosts and individual hosts.

The goal is to examine whether business-managed listings are associated with:
- different prices
- different guest ratings

Here, `biz = 1` represents Business Hosts, while `biz = 0` represents Individual Hosts.

In [33]:
biz_summary = (
    df_filtered.groupby("biz")
    .agg(
        median_price=("price", "median"),
        median_rating=("rating", "median"),
        listings=("price", "size")
    )
    .reset_index()
)

biz_summary["biz"] = biz_summary["biz"].map({
    0: "Individual Host",
    1: "Business Host"
})

biz_summary

,biz,median_price,median_rating,listings
0,Individual Host,195.917986,96.0,31489
1,Business Host,213.180113,92.0,16556


In [34]:
fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Host Type", "Median Price (€)", "Median Rating", "Listings"],
        fill_color="#1f3b73",
        font=dict(color="white", size=14),
        align="center"
    ),
    cells=dict(
        values=[
            biz_summary["biz"],
            biz_summary["median_price"].round(2),
            biz_summary["median_rating"].round(0),
            biz_summary["listings"]
        ],
        fill_color="white",
        align="center",
        font=dict(size=13)
    )
)])

fig.update_layout(
    title="Business Host vs Individual Host",
    title_x=0.5,
    template="plotly_white",
    width=1000,
    height=350
)

fig.show()



## Final Conclusion

Airbnb pricing is influenced primarily by location, property size, and market demand rather than customer satisfaction.

While premium cities such as Amsterdam and Paris command higher prices, they do not necessarily deliver better guest experiences.

Budget markets, such as Athens and Budapest, provide strong value for money, indicating that lower prices do not imply lower quality.

Additionally, weekend pricing behavior varies across cities, reflecting differences in travel demand patterns.